# Notebook 3: DistilBERT fine-tuning

This notebook uses these .csv files:
- `headlines_train.csv`
- `headline_validation.csv`
- `headline_test.csv`

It trains or loads an explicitly selected fine-tuned model version. It reports preliminary metrics on the validation set, then also evaluates the test set if `RUN_TEST_EVALUATION` is enabled. 

Required columns: `headline` and `label`, with `0 = non-clickbait/news`, `1 = clickbait`.


## 1. Choose a GPU runtime in Colab

In Colab, select **Runtime → Change runtime type → GPU** if one is available. GPU availability varies. Training can run on CPU, but it will be slower.


In [ ]:
%pip -q install transformers==4.50.3 accelerate datasets scikit-learn seaborn


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Mount Google Drive and choose train or load

Drive is mounted as a folder path in Colab; it is not a URL. Put all three CSV files in `MyDrive/clickbait_project/datasets/`.

- First run: keep `MODEL_ACTION = 'train'`. It starts from the pretrained DistilBERT model, fine-tunes it, and saves the result to Drive.
- Later/demo run: set `MODEL_ACTION = 'load'` to load that saved version without training.
- To make an updated model, choose a new version name (for example `distilbert_v2`) and set action to `train`. The notebook will not overwrite an existing model version.


In [ ]:
# Choose where this notebook reads data and saves/loads models.
STORAGE_MODE = 'drive'  # Choose exactly: 'drive' (Colab) or 'local' (Jupyter on your computer)

if STORAGE_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/clickbait_project')
elif STORAGE_MODE == 'local':
    # Expected local layout: project/datasets, project/models, project/results, project/notebooks.
    # If Jupyter starts in the notebooks folder, use its parent as the project folder.
    PROJECT_DIR = Path.cwd().resolve()
    if PROJECT_DIR.name.lower() == 'notebooks':
        PROJECT_DIR = PROJECT_DIR.parent
    # If your project is elsewhere, replace the line above with an explicit path, e.g.:
    # PROJECT_DIR = Path(r'C:\Users\YourName\clickbait-project')
else:
    raise ValueError("STORAGE_MODE must be 'drive' or 'local'.")

DATA_DIR = PROJECT_DIR / 'datasets'
MODELS_DIR = PROJECT_DIR / 'models'
RESULTS_DIR = PROJECT_DIR / 'results'
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Could not find datasets folder: {DATA_DIR}. Check STORAGE_MODE/PROJECT_DIR.')

MODEL_ACTION = 'train'       # Choose exactly: 'train' or 'load'
MODEL_VERSION = 'distilbert_v1'
MODEL_DIR = MODELS_DIR / MODEL_VERSION

# Shared files. Names are intentionally explicit; change only if your shared files use different names.
TRAIN_PATH = DATA_DIR / 'headlines_train.csv'
VALIDATION_PATH = DATA_DIR / 'headline_validation.csv'
TEST_PATH = DATA_DIR / 'headline_test.csv'

RUN_TEST_EVALUATION = True   # You asked to observe test metrics after validation.

if MODEL_ACTION not in {'train', 'load'}:
    raise ValueError("MODEL_ACTION must be 'train' or 'load'.")
if MODEL_ACTION == 'train' and MODEL_DIR.exists():
    raise FileExistsError(f'{MODEL_DIR} already exists. Choose a new MODEL_VERSION so an older model is preserved.')
if MODEL_ACTION == 'load' and not MODEL_DIR.exists():
    raise FileNotFoundError(f'No saved model at {MODEL_DIR}. Train that version first or correct MODEL_VERSION.')

for path in (TRAIN_PATH, VALIDATION_PATH):
    if not path.exists():
        raise FileNotFoundError(f'Missing {path.name}. Put it in {DATA_DIR}.')
if RUN_TEST_EVALUATION and not TEST_PATH.exists():
    raise FileNotFoundError(f'Missing {TEST_PATH.name}. Put it in {DATA_DIR}, or set RUN_TEST_EVALUATION = False.')

print('Action:', MODEL_ACTION)
print('Model version:', MODEL_VERSION)
print('Model path:', MODEL_DIR)


## 3. Load and check the splits

This cell verifies column names and binary labels. It does not combine or reshuffle the files. Both model notebooks should use these same train and validation files.


In [ ]:
def read_split(path, split_name):
    frame = pd.read_csv(path, encoding='utf-8-sig')
    required = {'headline', 'label'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
    frame = frame[['headline', 'label']].dropna().copy()
    frame['headline'] = frame['headline'].astype(str).str.strip()
    frame = frame[frame['headline'].ne('')].copy()
    frame['label'] = pd.to_numeric(frame['label'], errors='raise').astype(int)
    if not set(frame['label'].unique()).issubset({0, 1}):
        raise ValueError(f'{path.name} labels must be 0/1; found {sorted(frame["label"].unique())}')
    frame = frame.reset_index(drop=True)
    print(f'{split_name}: {len(frame):,} rows')
    display(frame['label'].value_counts().sort_index().rename_axis('label').to_frame('rows'))
    return frame

train_df = read_split(TRAIN_PATH, 'Training split')
validation_df = read_split(VALIDATION_PATH, 'Validation split')
test_df = read_split(TEST_PATH, 'Test split') if RUN_TEST_EVALUATION else None
print('Label mapping: 0 = non-clickbait/news; 1 = clickbait')


## 4. Tokenize text and load the model

Fine-tuning starts with the existing pretrained DistilBERT weights and adapts them to the two labels. The model is not trained from scratch. Headline length and training settings below are intentionally modest and fixed.


In [ ]:
BASE_MODEL = 'distilbert/distilbert-base-uncased'
MAX_LENGTH = 64
BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 2e-5

if MODEL_ACTION == 'train':
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=2,
        id2label={0: 'non-clickbait/news', 1: 'clickbait'},
        label2id={'non-clickbait/news': 0, 'clickbait': 1},
    )
else:
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR), local_files_only=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def make_hf_dataset(frame):
    raw = Dataset.from_pandas(
        pd.DataFrame({'text': frame['headline'], 'labels': frame['label']}),
        preserve_index=False,
    )
    return raw.map(
        lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=['text'],
    )

train_hf = make_hf_dataset(train_df)
validation_hf = make_hf_dataset(validation_df)
test_hf = make_hf_dataset(test_df) if RUN_TEST_EVALUATION else None
print('Tokenization complete.')


## 5. Train and save, or load the selected version

When `MODEL_ACTION` is `train`, the trainer sees only `headlines_train.csv`. Validation and test rows are not used to update weights. After training, the model and tokenizer are saved to the versioned Drive folder.


In [ ]:
training_args = TrainingArguments(
    output_dir='/content/distilbert_training_temp',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy='no',
    report_to='none',
    seed=42,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf if MODEL_ACTION == 'train' else None,
    data_collator=data_collator,
    processing_class=tokenizer,
)

if MODEL_ACTION == 'train':
    train_result = trainer.train()
    print('Training complete.')
    print(train_result.metrics)
    MODEL_DIR.mkdir(parents=True, exist_ok=False)
    trainer.save_model(str(MODEL_DIR))
    tokenizer.save_pretrained(str(MODEL_DIR))
    print('Saved model and tokenizer to:', MODEL_DIR)
else:
    print('Loaded saved model; training was skipped.')


## 6. Evaluate on validation (preliminary results)

These are the metrics to use for preliminary results and model improvements. The clickbait class is label 1. Inspect false positives and false negatives below.


In [ ]:
def evaluate_split(frame, hf_dataset, split_name):
    output = trainer.predict(hf_dataset)
    probs = torch.softmax(torch.tensor(output.predictions), dim=1).cpu().numpy()
    y_true = frame['label'].to_numpy()
    y_pred = probs.argmax(axis=1)
    clickbait_probability = probs[:, 1]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average='binary', zero_division=0
    )
    metrics = {
        'split': split_name,
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'clickbait_precision': float(precision),
        'clickbait_recall': float(recall),
        'clickbait_f1': float(f1),
        'rows': int(len(frame)),
        'model_version': MODEL_VERSION,
    }
    predictions = frame.copy()
    predictions['predicted_label'] = y_pred
    predictions['clickbait_confidence_percent'] = (clickbait_probability * 100).round(2)
    print(json.dumps(metrics, indent=2))
    print(classification_report(
        y_true, y_pred, labels=[0, 1],
        target_names=['non-clickbait/news', 'clickbait'], zero_division=0
    ))
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted non-clickbait', 'Predicted clickbait'],
                yticklabels=['Actual non-clickbait', 'Actual clickbait'])
    plt.title(f'{split_name.title()} confusion matrix')
    plt.xlabel('Prediction')
    plt.ylabel('Actual label')
    plt.tight_layout()
    plt.show()
    return metrics, predictions

validation_metrics, validation_predictions = evaluate_split(
    validation_df, validation_hf, 'validation'
)


In [ ]:
validation_errors = validation_predictions[
    validation_predictions['label'] != validation_predictions['predicted_label']
]
print('Validation errors:', len(validation_errors))
display(validation_errors[['headline', 'label', 'predicted_label', 'clickbait_confidence_percent']].head(20))


## 7. Optional test-set observation

This cell evaluates `headline_test.csv` when `RUN_TEST_EVALUATION = True`. The test file has already been accessed in prior work, and this run accesses it again. Treat these numbers as observational; do not use them to choose settings or make improvements. Use validation results for that.


In [ ]:
test_metrics = None
if RUN_TEST_EVALUATION:
    test_metrics, test_predictions = evaluate_split(test_df, test_hf, 'test')
    test_errors = test_predictions[test_predictions['label'] != test_predictions['predicted_label']]
    print('Test errors:', len(test_errors))
    display(test_errors[['headline', 'label', 'predicted_label', 'clickbait_confidence_percent']].head(20))
else:
    print('Test evaluation skipped.')


## 8. Save metrics and predictions to Drive

Validation results are saved for your preliminary presentation. Test results are also saved if the optional test evaluation is enabled. Confidence is the model's estimated clickbait probability, not a verified measure of how misleading a headline is.


In [ ]:
RESULT_VERSION_DIR = RESULTS_DIR / MODEL_VERSION
RESULT_VERSION_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULT_VERSION_DIR / 'validation_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(validation_metrics, f, indent=2)
validation_predictions.to_csv(
    RESULT_VERSION_DIR / 'validation_predictions.csv', index=False, encoding='utf-8-sig'
)

if test_metrics is not None:
    with open(RESULT_VERSION_DIR / 'test_metrics_observational.json', 'w', encoding='utf-8') as f:
        json.dump(test_metrics, f, indent=2)
    test_predictions.to_csv(
        RESULT_VERSION_DIR / 'test_predictions_observational.csv', index=False, encoding='utf-8-sig'
    )

print('Saved results to:', RESULT_VERSION_DIR)


## Settings used in this run

- Base model: `distilbert/distilbert-base-uncased`
- Training epochs: 2
- Maximum headline length: 64 tokens
- Batch size: 16
- Learning rate: 2e-5
- Random seed: 42
- Train split: `headlines_train.csv`
- Preliminary evaluation: `headline_validation.csv`
- Optional observation: `headline_test.csv`

For another training experiment, use `MODEL_ACTION = 'train'` and a new version name such as `distilbert_v2`. The notebook will preserve the earlier Drive model.
